# 03 - Does writing tables value by value help retrieval?

Same filings, same questions, same models. Only one thing changes: how tables are written in the chunks
(`text/` = cells flattened into a run of numbers, `tables/` = every value with its row and column).
The question is only about **finding the right passages** (no language model, no answer grading here).

How the comparison is made:
* the gold evidence of each question points to chunks of the text version; for the table version it is
  translated by matching content (`src/eval/remap.py`). It is an approximation, so the questions where the
  match was weak are listed and the result is also shown without them;
* three retrieval settings, from simple to complete: dense only; filing filter + reranker; filing filter +
  keyword search + reranker;
* the reranker limit is 1024 tokens for both versions, so no table chunk is cut.

**Before running:** Runtime > T4 GPU; Colab Secret `HF_TOKEN`. Needs notebooks `02a` and `02b` to be done.

In [ ]:
import os
import subprocess
import sys

from google.colab import userdata

REPO = "10K-agentic-rag"
BRANCH = "main"
try:  # only needed while the repository is private (a read-only token for this one repository)
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001  the secret does not exist: the repository is public
    token = None
login = f"{token}@" if token else ""
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://{login}github.com/ilaria-cesa/{REPO}.git"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never hard-code the token
HF_REPO = "IlariaCesa/10k-agentic-rag-data"

Load both versions of the chunks and vectors from the dataset, and the 175 questions from the repository.

In [ ]:
import json

from src.eval.golden import load_golden, validate
from src.eval.remap import remap_questions
from src.hub.dataset import download_folder
from src.retrieval.embed import load_chunks

download_folder(HF_REPO, "text", Path("data"))
download_folder(HF_REPO, "tables", Path("data"))
VARIANTS = {"text": "data/text", "tables": "data/tables"}
chunks = {
    "text": load_chunks("data/text/chunks_1600.jsonl"),
    "tables": load_chunks("data/tables/chunks_tables_1600.jsonl"),
}
vectors = {v: np.load(f"{p}/embeddings_bge-m3.npy").astype(np.float32) for v, p in VARIANTS.items()}
for v in VARIANTS:
    assert len(chunks[v]) == len(vectors[v]), v
    print(v, vectors[v].shape)

all_questions = load_golden(Path("questions/questions.jsonl"))
errors, _ = validate(all_questions, chunks["text"])
assert not errors, errors
answerable = [q for q in all_questions if q["expected"] == "answer"]
gold = {"text": answerable}
gold["tables"], weak = remap_questions(answerable, chunks["text"], chunks["tables"])
print(len(answerable), "answerable questions; weak table matches:", len(weak), weak)
print(
    "gold chunks per question (text / tables):",
    round(np.mean([len(q["evidence_chunk_ids"]) for q in gold["text"]]), 2),
    "/",
    round(np.mean([len(q["evidence_chunk_ids"]) for q in gold["tables"]]), 2),
)

Models: the same bge-m3 embeds the questions, and the same reranker reorders candidates for both versions.

In [ ]:
from sentence_transformers import SentenceTransformer

from src.retrieval.embed import embed_texts
from src.retrieval.pipeline import Corpus, RetrievalConfig, retrieve
from src.retrieval.rerank import Reranker

model = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
model.max_seq_length = 1024
query_vectors = embed_texts(model, [q["question"] for q in answerable], batch_size=32)
reranker = Reranker(max_length=1024)
corpora = {v: Corpus(chunks[v], vectors[v]) for v in VARIANTS}

Run the three settings on both versions. For every question we keep the ranked chunk ids.

In [ ]:
from src.eval.metrics import retrieval_metrics

CONFIGS = [
    RetrievalConfig("dense only"),
    RetrievalConfig("filter + rerank", use_filter=True, rerank=True),
    RetrievalConfig("filter + hybrid + rerank", use_filter=True, hybrid=True, rerank=True),
]
rows = []
for variant in VARIANTS:
    for config in CONFIGS:
        for q, vec in zip(gold[variant], query_vectors):
            idx, _ = retrieve(corpora[variant], q["question"], vec, config, reranker)
            ids = [chunks[variant][i]["chunk_id"] for i in idx]
            rows.append(
                {"variant": variant, "config": config.name, "id": q["id"], "split": q["split"],
                 "type": q["type"], "weak": q["id"] in weak, "ranked": ids}
                | retrieval_metrics(q["evidence_chunk_ids"], ids)
            )
df = pd.DataFrame(rows)
cols = ["hit@1", "hit@5", "hit@10", "recall@5", "all@5", "mrr"]
print(df.groupby(["config", "variant"])[cols].mean().round(3).to_string())

The same, without the questions whose table match was weak, and by question type.

In [ ]:
strong = df[~df["weak"]]
print("Without weak matches:", strong["id"].nunique(), "questions")
print(strong.groupby(["config", "variant"])[cols].mean().round(3).to_string())
for metric in ["hit@5", "mrr"]:
    print("\n" + metric + " by question type (filter + hybrid + rerank)")
    sub = df[df["config"] == "filter + hybrid + rerank"]
    print(sub.groupby(["type", "variant"])[metric].mean().unstack().round(3).to_string())
print("\nhit@5 by split (filter + hybrid + rerank)")
print(sub.groupby(["split", "variant"])["hit@5"].mean().unstack().round(3).to_string())

Paired view: for each question, does the table version find the evidence in the top 5 when the text version does not
(and the other way round)? Same questions, so this is a fairer test than comparing two averages.

In [ ]:
from math import comb

def sign_test(better, worse):
    n = better + worse
    if n == 0:
        return 1.0
    k = min(better, worse)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2**n)

for config in [c.name for c in CONFIGS]:
    sub = df[df["config"] == config].pivot(index="id", columns="variant", values="hit@5")
    better = int(((sub["tables"] == 1) & (sub["text"] == 0)).sum())
    worse = int(((sub["tables"] == 0) & (sub["text"] == 1)).sum())
    print(f"{config:28s} tables better on {better}, text better on {worse}, sign test p={sign_test(better, worse):.3f}")

Save the per-question ranked ids to look at the questions that still fail.

In [ ]:
os.makedirs("results", exist_ok=True)
df.to_json("results/retrieval_text_vs_tables.jsonl", orient="records", lines=True)
import shutil

from google.colab import files

shutil.make_archive("retrieval_text_vs_tables", "zip", "results")
files.download("retrieval_text_vs_tables.zip")

Copy back to the chat: the printed tables. The zip has the ranked chunk ids of every question and setting.